In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

# ── Start Spark Session ────────────────────────────────────────────────────
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-gold") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ── Paths ──────────────────────────────────────────────────────────────────
SILVER = "/home/jovyan/work/output/silver"
GOLD   = "/home/jovyan/work/output/gold"

os.makedirs(GOLD, exist_ok=True)

print("✅ Spark session started")
print(f"   Silver path : {SILVER}")
print(f"   Gold path   : {GOLD}")

✅ Spark session started
   Silver path : /home/jovyan/work/output/silver
   Gold path   : /home/jovyan/work/output/gold


In [2]:
# ── Read All Silver Tables ─────────────────────────────────────────────────
# We read from silver (clean data) not bronze (raw data)
# This is the correct medallion architecture pattern

orders    = spark.read.csv(f"{SILVER}/orders",      header=True, inferSchema=True)
items     = spark.read.csv(f"{SILVER}/order_items", header=True, inferSchema=True)
customers = spark.read.csv(f"{SILVER}/customers",   header=True, inferSchema=True)
products  = spark.read.csv(f"{SILVER}/products",    header=True, inferSchema=True)
sellers   = spark.read.csv(f"{SILVER}/sellers",     header=True, inferSchema=True)
payments  = spark.read.csv(f"{SILVER}/payments",    header=True, inferSchema=True)

print("✅ All silver tables loaded")
print(f"   orders      : {orders.count():>6} rows")
print(f"   order_items : {items.count():>6} rows")
print(f"   customers   : {customers.count():>6} rows")
print(f"   products    : {products.count():>6} rows")
print(f"   sellers     : {sellers.count():>6} rows")
print(f"   payments    : {payments.count():>6} rows")

✅ All silver tables loaded
   orders      :  12824 rows
   order_items :  15590 rows
   customers   :  53441 rows
   products    :  32951 rows
   sellers     :   3095 rows
   payments    : 103886 rows


In [3]:
# ── dim_customers ──────────────────────────────────────────────────────────
# Grain: one row per UNIQUE PERSON (deduplicated on customer_unique_id)
#
# The tricky part here:
# - customer_id changes every order (same person = different customer_id)
# - customer_unique_id stays the same for the same person across all orders
# - We need the city/state from their MOST RECENT order
# - We need aggregates (total orders, total spend) across ALL their orders

# Step 1: Join customers with orders to get purchase timestamps
cust_with_orders = (customers
    .join(
        orders.select("customer_id", "purchase_timestamp"),
        "customer_id"
    )
)

# Step 2: Window to find the most recent order per unique customer
# We use this to pick city/state from their latest order
w_latest = Window \
    .partitionBy("customer_unique_id") \
    .orderBy(F.desc("purchase_timestamp"))

cust_latest_location = (cust_with_orders
    .withColumn("rn", F.row_number().over(w_latest))
    .filter("rn = 1")
    .select(
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix"
    )
)

# Step 3: Calculate aggregates per unique customer
cust_agg = (customers
    .join(orders.select("customer_id", "order_id", "purchase_timestamp"), "customer_id")
    .join(items.select("order_id", "price", "freight_value"), "order_id")
    .groupBy("customer_unique_id")
    .agg(
        F.min("purchase_timestamp").alias("first_order_date"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum(F.col("price") + F.col("freight_value")).alias("total_spend")
    )
    .withColumn("is_repeat_customer", F.col("total_orders") > 1)
)

# Step 4: Join location + aggregates together
# Generate surrogate key using hash of customer_unique_id
# Hash is deterministic — same input always gives same output
# This means re-running the pipeline gives the same keys every time
dim_customers = (cust_latest_location
    .join(cust_agg, "customer_unique_id")
    .withColumn("customer_key",
        F.abs(F.hash("customer_unique_id")).cast(LongType()))
    .select(
        "customer_key",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
        "first_order_date",
        "total_orders",
        "total_spend",
        "is_repeat_customer"
    )
)

# Save to gold
dim_customers.write.mode("overwrite").option("header", True).csv(f"{GOLD}/dim_customers")

# Summary
total = dim_customers.count()
repeat = dim_customers.filter("is_repeat_customer = true").count()
print(f"✅ dim_customers saved")
print(f"   Total unique customers : {total}")
print(f"   Repeat customers       : {repeat}")
print(f"   One-time customers     : {total - repeat}")

✅ dim_customers saved
   Total unique customers : 6740
   Repeat customers       : 43
   One-time customers     : 6697


In [4]:
# ── dim_products ───────────────────────────────────────────────────────────
# Grain: one row per product
#
# Key thing here:
# product_volume_cm3 is a CALCULATED column
# length × height × width = volume
# If ANY dimension is missing we set volume to null
# We never guess or fill with zeros — that would be wrong data

dim_products = (products
    # Calculate volume only if all three dimensions exist
    .withColumn("product_volume_cm3",
        F.when(
            F.col("product_length_cm").isNotNull() &
            F.col("product_height_cm").isNotNull() &
            F.col("product_width_cm").isNotNull(),
            F.col("product_length_cm") *
            F.col("product_height_cm") *
            F.col("product_width_cm")
        ).otherwise(F.lit(None))
    )
    # Generate surrogate key using hash of product_id
    .withColumn("product_key",
        F.abs(F.hash("product_id")).cast(LongType()))
    .select(
        "product_key",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_volume_cm3",
        "product_photos_qty",
        F.col("product_description_lenght").alias("product_description_length")
    )
)

# Save to gold
dim_products.write.mode("overwrite").option("header", True).csv(f"{GOLD}/dim_products")

# Summary
total = dim_products.count()
null_volume = dim_products.filter(F.col("product_volume_cm3").isNull()).count()
categories = dim_products.select("product_category_name").distinct().count()
print(f"✅ dim_products saved")
print(f"   Total products         : {total}")
print(f"   Products without volume: {null_volume}")
print(f"   Unique categories      : {categories}")

✅ dim_products saved
   Total products         : 32951
   Products without volume: 2
   Unique categories      : 74


In [5]:
# ── dim_sellers ────────────────────────────────────────────────────────────
# Grain: one row per seller
# This is the simplest dimension — just add a surrogate key

dim_sellers = (sellers
    .withColumn("seller_key",
        F.abs(F.hash("seller_id")).cast(LongType()))
    .select(
        "seller_key",
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix"
    )
)

# Save to gold
dim_sellers.write.mode("overwrite").option("header", True).csv(f"{GOLD}/dim_sellers")

# Summary
total = dim_sellers.count()
states = dim_sellers.select("seller_state").distinct().count()
print(f"✅ dim_sellers saved")
print(f"   Total sellers  : {total}")
print(f"   Unique states  : {states}")
print(f"\n   Top 5 states by seller count:")
dim_sellers.groupBy("seller_state") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(5) \
    .show()

✅ dim_sellers saved
   Total sellers  : 3095
   Unique states  : 23

   Top 5 states by seller count:
+------------+-----+
|seller_state|count|
+------------+-----+
|          SP| 1849|
|          PR|  349|
|          MG|  244|
|          SC|  190|
|          RJ|  171|
+------------+-----+



In [6]:
# ── Payment Aggregations ───────────────────────────────────────────────────
# One order can have multiple payment rows (e.g. credit card + voucher)
# We need to aggregate payments to order level for the fact table
#
# Three things we need per order:
# 1. total_payment_value — sum of all payment rows
# 2. payment_type — the type with the HIGHEST value (alphabetically if tied)
# 3. payment_installments — the MAXIMUM installments across all payment rows

# 1. Total payment value per order
pay_total = (payments
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("total_payment_value"))
)

# 2. Payment type with highest value per order
# If tied — take alphabetically first (e.g. "boleto" before "credit_card")
w_pay_type = Window \
    .partitionBy("order_id") \
    .orderBy(F.desc("payment_value"), F.asc("payment_type"))

pay_type = (payments
    .withColumn("rn", F.row_number().over(w_pay_type))
    .filter("rn = 1")
    .select("order_id", "payment_type")
)

# 3. Maximum installments per order
pay_installments = (payments
    .groupBy("order_id")
    .agg(F.max("payment_installments").alias("payment_installments"))
)

# Join all three together
pay_agg = (pay_total
    .join(pay_type, "order_id")
    .join(pay_installments, "order_id")
)

print(f"✅ Payment aggregations ready")
print(f"   Orders with payment data : {pay_agg.count()}")
print(f"\n   Sample payment aggregations:")
pay_agg.show(5)

✅ Payment aggregations ready
   Orders with payment data : 99440

   Sample payment aggregations:
+--------------------+-------------------+------------+--------------------+
|            order_id|total_payment_value|payment_type|payment_installments|
+--------------------+-------------------+------------+--------------------+
|00018f77f2f0320c5...|             259.83| credit_card|                   3|
|00042b26cf59d7ce6...|             218.04| credit_card|                   3|
|00054e8431b9d7675...|              31.75| credit_card|                   1|
|0006ec9db01a64e59...|              97.32| credit_card|                   4|
|000aed2e25dbad2f9...|             152.77| credit_card|                   1|
+--------------------+-------------------+------------+--------------------+
only showing top 5 rows



In [7]:
# ── fact_order_items ───────────────────────────────────────────────────────
# Grain: one row per item sold in an order
#
# Most complex part: proportional payment distribution
# If an order has 3 items priced $10, $20, $70:
# Total = $100
# Item 1 gets 10% of total payment
# Item 2 gets 20% of total payment
# Item 3 gets 70% of total payment

# Step 1: Calculate total price per order for proportion calculation
item_order_total = (items
    .groupBy("order_id")
    .agg(F.sum("price").alias("order_price_total"))
)

# Step 2: Join customers to get customer_unique_id
# We need this to look up customer_key in dim_customers
cust_lookup = customers.select("customer_id", "customer_unique_id")

# Step 3: Build the fact table
fact = (items
    # Join orders to get order-level info
    .join(
        orders.select(
            "order_id",
            "customer_id",
            "order_status",
            "purchase_timestamp",
            "delivered_customer_date",
            "estimated_delivery_date"
        ),
        "order_id"
    )
    # Join item order totals for proportion calculation
    .join(item_order_total, "order_id")

    # Join payment aggregations
    .join(pay_agg, "order_id", "left")

    # Calculate proportional payment value per item
    # Each item gets (its price / total order price) × total payment
    .withColumn("payment_value",
        F.round(
            F.col("total_payment_value") *
            (F.col("price") / F.col("order_price_total")),
            2
        )
    )

    # Calculate delivery metrics
    .withColumn("days_to_deliver",
        F.when(
            F.col("delivered_customer_date").isNotNull(),
            F.datediff(
                F.col("delivered_customer_date"),
                F.col("purchase_timestamp")
            )
        ).otherwise(F.lit(None))
    )
    .withColumn("days_delivery_vs_estimate",
        F.when(
            F.col("delivered_customer_date").isNotNull(),
            F.datediff(
                F.col("delivered_customer_date"),
                F.col("estimated_delivery_date")
            )
        ).otherwise(F.lit(None))
    )
    # Positive = late, Negative = early, Null = not delivered yet
    .withColumn("is_late_delivery",
        F.when(
            F.col("days_delivery_vs_estimate").isNotNull(),
            F.col("days_delivery_vs_estimate") > 0
        ).otherwise(F.lit(None))
    )

    # Join customer_unique_id to look up customer_key
    .join(cust_lookup, "customer_id")
    .join(
        dim_customers.select("customer_unique_id", "customer_key"),
        "customer_unique_id"
    )

    # Join product_key
    .join(
        dim_products.select("product_id", "product_key"),
        "product_id"
    )

    # Join seller_key
    .join(
        dim_sellers.select("seller_id", "seller_key"),
        "seller_id"
    )

    # order_date is just the date part of purchase_timestamp
    .withColumn("order_date", F.to_date("purchase_timestamp"))

    # Surrogate key = hash of order_id + order_item_id
    .withColumn("order_item_sk",
        F.abs(F.hash("order_id", "order_item_id")).cast(LongType()))

    # Select only the columns defined in the schema
    .select(
        "order_item_sk",
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "seller_key",
        "order_date",
        "order_status",
        "price",
        "freight_value",
        "payment_value",
        "payment_type",
        "payment_installments",
        "days_to_deliver",
        "days_delivery_vs_estimate",
        "is_late_delivery"
    )
)

# Save to gold
fact.write.mode("overwrite").option("header", True).csv(f"{GOLD}/fact_order_items")

# Summary
total = fact.count()
delivered = fact.filter(F.col("days_to_deliver").isNotNull()).count()
late = fact.filter("is_late_delivery = true").count()
print(f"✅ fact_order_items saved")
print(f"   Total rows       : {total}")
print(f"   Delivered orders : {delivered}")
print(f"   Late deliveries  : {late}")
print(f"   Late rate        : {round(late/delivered*100, 1)}%")

✅ fact_order_items saved
   Total rows       : 7662
   Delivered orders : 7538
   Late deliveries  : 339
   Late rate        : 4.5%


In [8]:
# ── Gold Layer Validation ──────────────────────────────────────────────────
# This checks that our star schema is internally consistent
# No orphan keys = every customer_key, product_key, seller_key in the
# fact table exists in the corresponding dimension table
# If we find orphans it means our joins had a bug

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("🔍 Validating Gold Layer...")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# Read fact table fresh from disk to validate saved version
fact_check = spark.read.csv(f"{GOLD}/fact_order_items", header=True, inferSchema=True)
dim_cust   = spark.read.csv(f"{GOLD}/dim_customers",    header=True, inferSchema=True)
dim_prod   = spark.read.csv(f"{GOLD}/dim_products",     header=True, inferSchema=True)
dim_sell   = spark.read.csv(f"{GOLD}/dim_sellers",      header=True, inferSchema=True)

# Record counts
print("\n📊 Record counts:")
print(f"   fact_order_items : {fact_check.count():>6} rows")
print(f"   dim_customers    : {dim_cust.count():>6} rows")
print(f"   dim_products     : {dim_prod.count():>6} rows")
print(f"   dim_sellers      : {dim_sell.count():>6} rows")

# Orphan key checks
# left_anti join returns rows in fact that have NO match in dimension
orphan_cust = fact_check.join(
    dim_cust.select("customer_key"), "customer_key", "left_anti"
).count()

orphan_prod = fact_check.join(
    dim_prod.select("product_key"), "product_key", "left_anti"
).count()

orphan_sell = fact_check.join(
    dim_sell.select("seller_key"), "seller_key", "left_anti"
).count()

print("\n🔑 Orphan key checks:")
print(f"   Orphan customer_keys : {orphan_cust} {'✅' if orphan_cust == 0 else '❌'}")
print(f"   Orphan product_keys  : {orphan_prod} {'✅' if orphan_prod == 0 else '❌'}")
print(f"   Orphan seller_keys   : {orphan_sell} {'✅' if orphan_sell == 0 else '❌'}")

# Surrogate key uniqueness check
duplicate_sk = fact_check.groupBy("order_item_sk") \
    .count() \
    .filter("count > 1") \
    .count()

print(f"\n🔑 Surrogate key uniqueness:")
print(f"   Duplicate order_item_sk : {duplicate_sk} {'✅' if duplicate_sk == 0 else '❌'}")

# Null checks on foreign keys
null_cust = fact_check.filter(F.col("customer_key").isNull()).count()
null_prod = fact_check.filter(F.col("product_key").isNull()).count()
null_sell = fact_check.filter(F.col("seller_key").isNull()).count()

print(f"\n🔑 Null foreign key checks:")
print(f"   Null customer_keys : {null_cust} {'✅' if null_cust == 0 else '❌'}")
print(f"   Null product_keys  : {null_prod} {'✅' if null_prod == 0 else '❌'}")
print(f"   Null seller_keys   : {null_sell} {'✅' if null_sell == 0 else '❌'}")

print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
if all(x == 0 for x in [orphan_cust, orphan_prod, orphan_sell,
                          duplicate_sk, null_cust, null_prod, null_sell]):
    print("✅ ALL VALIDATION CHECKS PASSED")
    print("   Gold layer is clean and ready for analysis")
else:
    print("❌ SOME CHECKS FAILED — review above")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔍 Validating Gold Layer...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📊 Record counts:
   fact_order_items :   7662 rows
   dim_customers    :   6740 rows
   dim_products     :  32951 rows
   dim_sellers      :   3095 rows

🔑 Orphan key checks:
   Orphan customer_keys : 0 ✅
   Orphan product_keys  : 0 ✅
   Orphan seller_keys   : 0 ✅

🔑 Surrogate key uniqueness:
   Duplicate order_item_sk : 0 ✅

🔑 Null foreign key checks:
   Null customer_keys : 0 ✅
   Null product_keys  : 0 ✅
   Null seller_keys   : 0 ✅

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ ALL VALIDATION CHECKS PASSED
   Gold layer is clean and ready for analysis
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
